# Early Exit: LSTM vs ResNet18 on CIFAR-10

This notebook trains two early-exit models on CIFAR-10 and compares them:
- **Early Exit LSTM** — treats each image row as a time step (32 steps × 96 features)
- **Early Exit ResNet18** — pretrained backbone with 3 exit heads

Both models have **3 exit points** driven by a confidence threshold.
Final comparison covers accuracy, exit distribution, and FLOPs.

In [ ]:
# ✅ Install dependencies
!pip install -q torch torchvision thop

import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights
from thop import profile
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
# ✅ Data — two transforms: ResNet needs 224x224, LSTM uses 32x32
transform_resnet = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_lstm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# ResNet loaders
train_rn = torchvision.datasets.CIFAR10('./data', train=True,  download=True, transform=transform_resnet)
test_rn  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=transform_resnet)
train_loader_rn = torch.utils.data.DataLoader(train_rn, batch_size=64, shuffle=True,  num_workers=2)
test_loader_rn  = torch.utils.data.DataLoader(test_rn,  batch_size=64, shuffle=False, num_workers=2)

# LSTM loaders (32x32 original size)
train_lstm = torchvision.datasets.CIFAR10('./data', train=True,  download=False, transform=transform_lstm)
test_lstm  = torchvision.datasets.CIFAR10('./data', train=False, download=False, transform=transform_lstm)
train_loader_lstm = torch.utils.data.DataLoader(train_lstm, batch_size=64, shuffle=True,  num_workers=2)
test_loader_lstm  = torch.utils.data.DataLoader(test_lstm,  batch_size=64, shuffle=False, num_workers=2)

print('Data ready.')
print(f'CIFAR-10 image shape: {train_lstm[0][0].shape}  (C x H x W)')

Data ready.
CIFAR-10 image shape: torch.Size([3, 32, 32])  (C x H x W)


---
## Model 1 — Early Exit LSTM
CIFAR-10 image (3×32×32) is reshaped to a sequence of **32 time steps**, each of size **96** (3 channels × 32 pixels per row).
The LSTM has 3 stacked layers; exit heads are placed after layers 1, 2, and 3.

In [ ]:
class EarlyExitLSTM(nn.Module):
    """
    3-layer stacked LSTM with one exit head per layer.
    Input: (batch, 32, 96)  — 32 row-steps, 96 features (3 ch * 32 px)
    """
    def __init__(self, input_size=96, hidden_size=256, num_layers=3, num_classes=10):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # One LSTM per layer so we can tap intermediate hidden states
        self.lstm1 = nn.LSTM(input_size,   hidden_size, batch_first=True)
        self.lstm2 = nn.LSTM(hidden_size,  hidden_size, batch_first=True)
        self.lstm3 = nn.LSTM(hidden_size,  hidden_size, batch_first=True)

        # Exit heads — take last hidden state of each layer
        self.exit1 = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        self.exit2 = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.exit3 = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x: (batch, C, H, W) → (batch, H, C*W) = (batch, 32, 96)
        B, C, H, W = x.shape
        x = x.permute(0, 2, 1, 3).reshape(B, H, C * W)

        out1, _ = self.lstm1(x)           # (B, 32, 256)
        h1 = out1[:, -1, :]               # last time step

        out2, _ = self.lstm2(out1)
        h2 = out2[:, -1, :]

        out3, _ = self.lstm3(out2)
        h3 = out3[:, -1, :]

        return self.exit1(h1), self.exit2(h2), self.exit3(h3)


lstm_model = EarlyExitLSTM().to(device)
total_lstm = sum(p.numel() for p in lstm_model.parameters())
print(f'Early Exit LSTM total params: {total_lstm:,}')

Early Exit LSTM total params: 1,586,078


---
## Model 2 — Early Exit ResNet18

In [ ]:
class EarlyExitResNet(nn.Module):
    def __init__(self, base_model, num_classes=10):
        super().__init__()
        self.stem   = nn.Sequential(base_model.conv1, base_model.bn1,
                                     base_model.relu,  base_model.maxpool)
        self.layer1 = base_model.layer1
        self.layer2 = base_model.layer2
        self.layer3 = base_model.layer3
        self.layer4 = base_model.layer4

        self.exit1 = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )
        self.exit2 = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes)
        )
        self.final_exit = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(512, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x  = self.stem(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return self.exit1(x1), self.exit2(x2), self.final_exit(x4)


base = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in base.parameters(): p.requires_grad = False
for p in base.layer3.parameters(): p.requires_grad = True
for p in base.layer4.parameters(): p.requires_grad = True

resnet_model = EarlyExitResNet(base).to(device)
total_rn = sum(p.numel() for p in resnet_model.parameters())
trainable_rn = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f'Early Exit ResNet total params:     {total_rn:,}')
print(f'Early Exit ResNet trainable params: {trainable_rn:,}')

Early Exit ResNet total params:     11,489,502
Early Exit ResNet trainable params: 10,806,430


---
## Shared Training & Evaluation Utilities

In [ ]:
LOSS_WEIGHTS = (0.5, 0.3, 0.2)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    w1, w2, w3 = LOSS_WEIGHTS
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        o1, o2, o3 = model(inputs)
        loss = w1 * criterion(o1, labels) + w2 * criterion(o2, labels) + w3 * criterion(o3, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, threshold, device):
    model.eval()
    correct, total = 0, 0
    exit_counts  = [0, 0, 0]
    exit_correct = [0, 0, 0]

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            o1, o2, o3 = model(inputs)
            for i in range(inputs.size(0)):
                c1, p1 = torch.max(F.softmax(o1[i], dim=0), dim=0)
                if c1.item() >= threshold:
                    pred, eidx = p1, 0
                else:
                    c2, p2 = torch.max(F.softmax(o2[i], dim=0), dim=0)
                    if c2.item() >= threshold:
                        pred, eidx = p2, 1
                    else:
                        _, p3 = torch.max(F.softmax(o3[i], dim=0), dim=0)
                        pred, eidx = p3, 2
                ok = int(pred.item() == labels[i].item())
                correct += ok
                exit_correct[eidx] += ok
                exit_counts[eidx]  += 1
                total += 1

    return correct / total * 100, exit_counts, exit_correct


print('Utilities defined.')

Utilities defined.


---
## Train Both Models (30 epochs each)

In [ ]:
NUM_EPOCHS     = 30
TRAIN_THRESH   = 0.80
criterion      = nn.CrossEntropyLoss()

opt_lstm   = optim.Adam(lstm_model.parameters(),   lr=1e-3)
opt_resnet = optim.Adam(filter(lambda p: p.requires_grad, resnet_model.parameters()), lr=0.001)

sched_lstm   = optim.lr_scheduler.StepLR(opt_lstm,   step_size=10, gamma=0.5)
sched_resnet = optim.lr_scheduler.StepLR(opt_resnet, step_size=10, gamma=0.5)

hist_lstm   = {'loss': [], 'acc': [], 'exits': []}
hist_resnet = {'loss': [], 'acc': [], 'exits': []}

print('='*65)
print(f'{'Epoch':>5}  {'LSTM Loss':>10}  {'LSTM Acc':>9}  {'RN Loss':>8}  {'RN Acc':>7}')
print('='*65)

for epoch in range(NUM_EPOCHS):
    # --- LSTM ---
    loss_l = train_one_epoch(lstm_model,   train_loader_lstm, criterion, opt_lstm,   device)
    acc_l, exits_l, _ = evaluate(lstm_model, test_loader_lstm, TRAIN_THRESH, device)
    sched_lstm.step()

    # --- ResNet ---
    loss_r = train_one_epoch(resnet_model, train_loader_rn,   criterion, opt_resnet, device)
    acc_r, exits_r, _ = evaluate(resnet_model, test_loader_rn, TRAIN_THRESH, device)
    sched_resnet.step()

    hist_lstm['loss'].append(loss_l);   hist_lstm['acc'].append(acc_l);   hist_lstm['exits'].append(exits_l)
    hist_resnet['loss'].append(loss_r); hist_resnet['acc'].append(acc_r); hist_resnet['exits'].append(exits_r)

    print(f'{epoch+1:>5}  {loss_l:>10.4f}  {acc_l:>8.2f}%  {loss_r:>8.4f}  {acc_r:>6.2f}%')

print('='*65)
print('Training complete.')

Epoch   LSTM Loss   LSTM Acc   RN Loss   RN Acc
    1      1.7850     41.56%    1.6653   86.26%
    2      1.4985     47.83%    1.4273   88.32%
    3      1.3407     53.52%    1.3507   89.31%
    4      1.2147     56.17%    1.3001   89.19%
    5      1.1115     58.44%    1.2683   89.96%
    6      1.0170     58.13%    1.2430   90.08%
    7      0.9215     58.51%    1.2204   90.37%
    8      0.8335     59.59%    1.2058   90.48%
    9      0.7489     59.20%    1.1868   89.82%
   10      0.6655     59.64%    1.1735   89.74%
   11      0.4761     60.31%    1.1503   91.37%
   12      0.3797     60.34%    1.1430   90.75%
   13      0.3174     60.15%    1.1383   90.57%
   14      0.2759     60.19%    1.1331   90.64%
   15      0.2474     59.72%    1.1282   90.43%
   16      0.2126     59.08%    1.1249   90.33%
   17      0.1934     60.02%    1.1214   90.75%
   18      0.1687     59.81%    1.1166   90.26%
   19      0.1651     59.50%    1.1143   90.15%
   20      0.1507     59.52%    1.1107  

---
## Threshold Sweep — Same Trained Models

In [ ]:
thresholds = [0.99, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50]
sweep_lstm   = {}
sweep_resnet = {}

print('\nLSTM Threshold Sweep')
print('-'*60)
for thr in thresholds:
    acc, exits, ecorr = evaluate(lstm_model, test_loader_lstm, thr, device)
    sweep_lstm[thr] = {'acc': acc, 'exits': exits, 'ecorr': ecorr}
    e1p = exits[0]/100; e2p = exits[1]/100
    print(f'  thr={thr:.2f} | Acc={acc:.2f}% | Exit1={exits[0]:5d} ({e1p:.1f}%) | Exit2={exits[1]:5d} ({e2p:.1f}%) | Exit3={exits[2]:5d}')

print('\nResNet Threshold Sweep')
print('-'*60)
for thr in thresholds:
    acc, exits, ecorr = evaluate(resnet_model, test_loader_rn, thr, device)
    sweep_resnet[thr] = {'acc': acc, 'exits': exits, 'ecorr': ecorr}
    e1p = exits[0]/100; e2p = exits[1]/100
    print(f'  thr={thr:.2f} | Acc={acc:.2f}% | Exit1={exits[0]:5d} ({e1p:.1f}%) | Exit2={exits[1]:5d} ({e2p:.1f}%) | Exit3={exits[2]:5d}')

---
## FLOPs Analysis

In [ ]:
# LSTM FLOPs — wrap each exit path
class LSTMExit1(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.lstm1 = m.lstm1; self.exit1 = m.exit1
    def forward(self, x):
        B,C,H,W = x.shape
        x = x.permute(0,2,1,3).reshape(B,H,C*W)
        o,_ = self.lstm1(x); return self.exit1(o[:,-1,:])

class LSTMExit2(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.lstm1=m.lstm1; self.lstm2=m.lstm2; self.exit2=m.exit2
    def forward(self, x):
        B,C,H,W = x.shape
        x = x.permute(0,2,1,3).reshape(B,H,C*W)
        o1,_=self.lstm1(x); o2,_=self.lstm2(o1); return self.exit2(o2[:,-1,:])

class LSTMExit3(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.lstm1=m.lstm1; self.lstm2=m.lstm2; self.lstm3=m.lstm3; self.exit3=m.exit3
    def forward(self, x):
        B,C,H,W = x.shape
        x = x.permute(0,2,1,3).reshape(B,H,C*W)
        o1,_=self.lstm1(x); o2,_=self.lstm2(o1); o3,_=self.lstm3(o2); return self.exit3(o3[:,-1,:])

dummy_lstm   = torch.randn(1, 3, 32,  32).to(device)
dummy_resnet = torch.randn(1, 3, 224, 224).to(device)

lf1,_ = profile(LSTMExit1(lstm_model).to(device), inputs=(dummy_lstm,),   verbose=False)
lf2,_ = profile(LSTMExit2(lstm_model).to(device), inputs=(dummy_lstm,),   verbose=False)
lf3,_ = profile(LSTMExit3(lstm_model).to(device), inputs=(dummy_lstm,),   verbose=False)

class RNExit1(nn.Module):
    def __init__(self,m): super().__init__(); self.stem=m.stem; self.l1=m.layer1; self.e=m.exit1
    def forward(self,x): return self.e(self.l1(self.stem(x)))

class RNExit2(nn.Module):
    def __init__(self,m): super().__init__(); self.stem=m.stem; self.l1=m.layer1; self.l2=m.layer2; self.e=m.exit2
    def forward(self,x): return self.e(self.l2(self.l1(self.stem(x))))

class RNExit3(nn.Module):
    def __init__(self,m): super().__init__(); self.stem=m.stem; self.l1=m.layer1; self.l2=m.layer2; self.l3=m.layer3; self.l4=m.layer4; self.e=m.final_exit
    def forward(self,x): return self.e(self.l4(self.l3(self.l2(self.l1(self.stem(x))))))

rf1,_ = profile(RNExit1(resnet_model).to(device), inputs=(dummy_resnet,), verbose=False)
rf2,_ = profile(RNExit2(resnet_model).to(device), inputs=(dummy_resnet,), verbose=False)
rf3,_ = profile(RNExit3(resnet_model).to(device), inputs=(dummy_resnet,), verbose=False)

print('FLOPs per sample at each exit point')
print('-'*55)
print(f'  LSTM   Exit1: {lf1/1e6:7.2f} MFLOPs  (saves {(1-lf1/lf3)*100:.1f}%)')
print(f'  LSTM   Exit2: {lf2/1e6:7.2f} MFLOPs  (saves {(1-lf2/lf3)*100:.1f}%)')
print(f'  LSTM   Exit3: {lf3/1e6:7.2f} MFLOPs  (full)')
print()
print(f'  ResNet Exit1: {rf1/1e6:7.2f} MFLOPs  (saves {(1-rf1/rf3)*100:.1f}%)')
print(f'  ResNet Exit2: {rf2/1e6:7.2f} MFLOPs  (saves {(1-rf2/rf3)*100:.1f}%)')
print(f'  ResNet Exit3: {rf3/1e6:7.2f} MFLOPs  (full)')

In [ ]:
# Expected average FLOPs per threshold
lstm_flops   = [lf1, lf2, lf3]
resnet_flops = [rf1, rf2, rf3]

print('Expected avg FLOPs per sample at each threshold')
print(f'  {"Threshold":>10} | {"LSTM MFLOPs":>12} | {"LSTM Savings":>13} | {"RN MFLOPs":>10} | {"RN Savings":>10}')
print('-'*70)
for thr in thresholds:
    le = sweep_lstm[thr]['exits'];   avg_l = sum(c*f for c,f in zip(le, lstm_flops))   / sum(le)
    re = sweep_resnet[thr]['exits']; avg_r = sum(c*f for c,f in zip(re, resnet_flops)) / sum(re)
    print(f'  {thr:>10.2f} | {avg_l/1e6:>12.2f} | {(1-avg_l/lf3)*100:>12.1f}% | {avg_r/1e6:>10.2f} | {(1-avg_r/rf3)*100:>9.1f}%')

---
## Final Visualizations

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
thr_vals = thresholds

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Early Exit: LSTM vs ResNet18 on CIFAR-10', fontsize=15, fontweight='bold')

# --- Row 0: Training curves ---
# Loss
axes[0,0].plot(epochs, hist_lstm['loss'],   label='LSTM',   color='royalblue',  linewidth=2)
axes[0,0].plot(epochs, hist_resnet['loss'], label='ResNet', color='tomato',     linewidth=2)
axes[0,0].set_title('Training Loss'); axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Accuracy
axes[0,1].plot(epochs, hist_lstm['acc'],   label='LSTM',   color='royalblue',  linewidth=2)
axes[0,1].plot(epochs, hist_resnet['acc'], label='ResNet', color='tomato',     linewidth=2)
axes[0,1].set_title(f'Test Accuracy (thr={TRAIN_THRESH})')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# Accuracy vs Threshold
axes[0,2].plot(thr_vals, [sweep_lstm[t]['acc']   for t in thr_vals], 'o-', label='LSTM',   color='royalblue', linewidth=2)
axes[0,2].plot(thr_vals, [sweep_resnet[t]['acc'] for t in thr_vals], 's-', label='ResNet', color='tomato',    linewidth=2)
axes[0,2].set_title('Accuracy vs Threshold')
axes[0,2].set_xlabel('Confidence Threshold'); axes[0,2].set_ylabel('Accuracy (%)')
axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

# --- Row 1: Exit distributions ---
# LSTM exit distribution over epochs (stackplot)
e1l = [e[0] for e in hist_lstm['exits']]
e2l = [e[1] for e in hist_lstm['exits']]
e3l = [e[2] for e in hist_lstm['exits']]
axes[1,0].stackplot(epochs, e1l, e2l, e3l,
                    labels=['Exit1','Exit2','Exit3'],
                    colors=['#2ecc71','#f39c12','#e74c3c'], alpha=0.8)
axes[1,0].set_title('LSTM Exit Distribution per Epoch')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Samples'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# ResNet exit distribution over epochs
e1r = [e[0] for e in hist_resnet['exits']]
e2r = [e[1] for e in hist_resnet['exits']]
e3r = [e[2] for e in hist_resnet['exits']]
axes[1,1].stackplot(epochs, e1r, e2r, e3r,
                    labels=['Exit1','Exit2','Exit3'],
                    colors=['#2ecc71','#f39c12','#e74c3c'], alpha=0.8)
axes[1,1].set_title('ResNet Exit Distribution per Epoch')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Samples'); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

# FLOPs savings vs threshold
lstm_savings   = [(1 - sum(c*f for c,f in zip(sweep_lstm[t]['exits'],   lstm_flops))   / sum(sweep_lstm[t]['exits'])   / lf3)*100 for t in thr_vals]
resnet_savings = [(1 - sum(c*f for c,f in zip(sweep_resnet[t]['exits'], resnet_flops)) / sum(sweep_resnet[t]['exits']) / rf3)*100 for t in thr_vals]

axes[1,2].plot(thr_vals, lstm_savings,   'o-', label='LSTM',   color='royalblue', linewidth=2)
axes[1,2].plot(thr_vals, resnet_savings, 's-', label='ResNet', color='tomato',    linewidth=2)
axes[1,2].set_title('FLOPs Savings vs Threshold')
axes[1,2].set_xlabel('Confidence Threshold'); axes[1,2].set_ylabel('FLOPs Saved (%)')
axes[1,2].legend(); axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_vs_resnet_early_exit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# ✅ Save both models
torch.save(lstm_model.state_dict(),   'early_exit_lstm_cifar10.pth')
torch.save(resnet_model.state_dict(), 'early_exit_resnet_cifar10.pth')
print('Both models saved.')